## Gateway Viability

- **Goal**: to understand the economic incentives to become and to remain a gateway
- **System Goals Targeted**:
  - Economic Viability
  - Accessibility
- **Design**: create different demand-side circumstances (e.g. application growth, relay per application growth) that impact gateway profitability
- **Testing mechanism**:
  - Introduce different rates of arrival for applications into the ecosystem
  - Introduce different relay traffic rates per application
  - Assess the impact of different `TokenToRelaysMultiplier` (`TTRM`) values on gateway profitability, comparing revenue from being a gateway to revenue from being a servicer (potentially making `TTRM` different _for each service_ )
  - Assess the impact of different gateway fees per relay (`GFPR`) on gateway profitability, comparing revenue from being a gateway to revenue from being a servicer
  - Assess different minimum staking amounts as a barrier to entry
- **KPIs**:
  - Servicer NPV (KPI-1)
  - Gateway NPV (KPI-3)
  - Servicer Capital Costs per Unit of Reward (KPI-14)
  - DAO value capture (KPI-10)

## Protocol Parameters to be swept

1. `TokenToRelaysMultiplier` (`TTRM`)
2. `GatewayFeePerRelay` (`GFPR`)
3. `ApplicationFeePerRelay` (`AFPR`, assumed different from `GFPR`)
4. `GatewayMinimumStake` (`GMS`)
5. `ApplicationMinimumStake` (`AMS`)

## Exogenous Parameters to be swept
1. Maximum number of applications in the simulation: [params["application_max_number"]](https://github.com/BlockScience/PocketSimulationModel/blob/scenario-notebooks/model/boundary_actions/application.py#L15): $A$. Sweeping this parameter changes the arrival rate of applications by adjusting the time before the network is 'saturated'.
2. Average arrival rate of the relay request process (cf. below): $\pi_0$

## Exogenous Distributions that are fixed for this scenario (but may depend upon a parameter being swept)

1. The probability distribution of relays requested by an application per day: $F^a(r; \boldsymbol \pi) \in \mathcal{F}$. Probability distribution (within the set of continuous probability distributions with bounded support $\mathcal{F}$) of the number of relays per application per day (a random variable). This distribution is parameterized by a finite number of parameters $\boldsymbol \pi$, $|\boldsymbol \pi| < \infty$. _One_ of these parameters, say $\pi_0$, represents the arrival rate of relays over the course of one day.

**For this scenario, the distribution is located in the function** `submit_relay_requests_ba_gamma()` [within model/boundary_actions/application_ba.py](https://github.com/BlockScience/PocketSimulationModel/blob/main/model/boundary_actions/application.py#L116) 

## Simulation Initialization

1. Initialize a set $N_s$ of servicers
2. Initialize a set $N_c$ of services (chains)
3. Initialize a set $N_a$ of applications
4. Initialize a set $N_g$ of gateways
5. Initialize the network of applications - gateways - services - servicers
6. Initialize the number of timesteps $T$

## Simulation routine

**Each simulation runs as the "default", with different maximum number of applications, average arrival rate of requests, and other sweep parameter values.**


## Full Grid Parameter Sweep Values (7 parameters in total)

|  Name | Sweep Values | Units |
| --- | ---| ---|
| `TTRM` | (25, 100, 400) | relay/uPOKT |
| `GFPR` | (20, 25, 30) | uPOKT/relay |
| `AFPR` | (20, 25, 30) | uPOKT/relay |
| `GMS`  | (1e5, 1.5e5, 2e5) | POKT |
| `AMS`  | (1e4, 1.5e4, 2e4)  | POKT |
| $A$  | (1, 20, 100) | number |
| $\pi_0$  | (250, 25000, 250,000) ? | relays/day |

**Total number of parameter constellations**: $3^{7} = 2,187$

**Total number of Monte Carlo runs per constellation**: ? 30

**Total number of experiments**: 65,610

## Adaptive Grid Parameter Sweep Values (7 parameters in total)

|  Name | Sweep Values | Units |
| --- | ---| ---|
| `TTRM` | (25, 400) | relay/uPOKT |
| `GFPR` | (20, 30) | uPOKT/relay |
| `AFPR` | (20, 30) | uPOKT/relay |
| `GMS`  | (1e5, 2e5) | POKT |
| `AMS`  | (1e4, 2e4)  | POKT |
| $A$  | (1, 20, 100) | number |
| $\pi_0$  | (250, 25000, 250,000) ? | relays/day |

**Total number of parameter constellations**: $2^{5} \times 3^2 = 288$

**Total number of Monte Carlo runs per constellation**: ? 30

**Total number of experiments**: 8,640

## Loop algorithm (as many times as is feasible given resource/time constraints)

1. Run experiments
2. Assess KPIs via threshold inequalities/sensitivity analysis
3. Select representative 'best' constellation
4. Generate new $(min,max)$ grid sweep values for each parameter around 'best' constellation values
5. Loop--back to #1.

In [2]:
from calc import * 


NameError: name 'run_experiments' is not defined

In [ ]:
def create_new_binary_sweep_individual(df: pd.DataFrame,
                          col_to_rank_by: str, 
                          keys: List[str]):
   """
   Given an individual (defined as a dictionary whose keys are two-element lists),
   we create a new sweep. 
   """
    best_index = df[col_to_rank_by].idmax() # location of best individual
    best_individual = df.loc[best_index] # find best individual
    best__score = df[col_to_rank_by].max() #best value for ranking
    new_sweep_individual = dict{} # create new sweep individual

    # Loop over the defined keys, in each case keeping the 
    # best individual's value as a bound,  and replacing the 
    # other bound with the average of the two bounds. 

    for key in keys:
        bounds = df[key].unique()
        if len(bounds) == 2:
            lb = min(bounds)
            ub = max(bounds)
            mid = 0.5 * (lb + ub)
            if isclose(best_individual[key],lb):
                new_sweep_individual[key] =  [lb, mid]
            elif isclose(best_individual[key], ub):
                new_sweep_individual_key = [mid, ub]
            else:
                raise Error("Something is wrong. Best individual has issues.")
        else:
            raise Error("Passed an individual with a list who didn't have two elements. ")
            
    return new_sweep_individual


def equal_on_keys(ind_1, ind_2, keys_to_use: list[Str]):
    """
    Check to see if two dictionary individuals are the same,
     i.e. if they take the
    same value for all relevant keys.
    """
    return all([isclose(ind_1.get(key),ind_2.get(key)) 
                for key in ind_1.keys()
                if key in keys_to_use])

In [ ]:
def iterate_adaptive_grid_psuu(initial_indvidual,
                               max_repeat_individual = 6,
                               max_repeat_threshold_score = 6,
                               max_steps = 100,
                               col_to_use = ""):
    step = 0
    keys_to_use = initial_individual.keys()

    last_best_individual = None
    last_best_threshold_score = None

    times_individual_repeated = 0
    times_threshold_score = 0

    done = False
    while(not(done)): 
        # Get information and convert to info on individuals
        # Find new individual
        # Check if we are done. 
        # If not, continue.
        # IF yes, indicate. 
        


In [16]:
config_option_map_sweep = {}

gateway_viability_sweep_ag1_ = build_params("Base")
gateway_viability_sweep_ag1_["relays_to_tokens_multiplier"] = [25, 400]
gateway_viability_sweep_ag1_["gateway_fee_per_relay"] = [20, 30]
gateway_viability_sweep_ag1_["application_fee_per_relay"] = [20, 30]
gateway_viability_sweep_ag1_["gateway_minimum_stake"] = [100000 * 1e6, 200000 * 1e6]
gateway_viability_sweep_ag1_["application_minimum_stake"] = [
    100000 * 1e6,
    200000 * 1e6,
]

gateway_viability_sweep_ag1_["application_max_number"] = [5, 20, 100]
gateway_viability_sweep_ag1_["relays_per_session_gamma_distribution_scale"] = [
    100000,
    300000,
    900000,
]

my_sweep = create_sweep(
    "gateway_viability_sweep_ag1_",
    gateway_viability_sweep_ag1_,
    config_option_map_sweep,
)

run_experiments(my_sweep)

NameError: name 'run_experiments' is not defined